In [6]:
import pandas as pd
from pathlib import Path

In [7]:
file_path = Path("RASFF_raw_data_EU.csv")

def perform_eda(file_path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc, low_memory=False)
            print(f"Loaded {file_path.name} with encoding={enc}")
            return df
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            try:
                df = pd.read_csv(
                    file_path,
                    encoding=enc,
                    engine="python",
                    on_bad_lines="skip",
                )
                print(
                    f"Loaded {file_path.name} with encoding={enc}, engine=python, on_bad_lines=skip "
                    "(some malformed rows may be skipped)"
                )
                return df
            except Exception:
                continue

    # Last fallback: keep pipeline alive by replacing undecodable bytes.
    df = pd.read_csv(
        file_path,
        encoding="utf-8",
        encoding_errors="replace",
        engine="python",
        on_bad_lines="skip",
    )
    print(
        f"Loaded {file_path.name} with encoding=utf-8 (invalid bytes replaced), "
        "engine=python, on_bad_lines=skip"
    )
    return df

df = perform_eda(file_path)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head(3)

Loaded RASFF_raw_data_EU.csv with encoding=utf-8, engine=python, on_bad_lines=skip (some malformed rows may be skipped)
Rows: 29,624 | Columns: 14


,reference,category,type,subject,date,notifying_country,classification,risk_decision,distribution,forAttention,forFollowUp,operator,origin,hazards
0,2026.2434,"dietetic foods, food supplements and fortified...",food,Unlabelled allergen soja in whey protein isolate,20-03-2026 18:01:28,Switzerland,alert notification,serious,Switzerland,NaN,Germany,"Germany,Switzerland",Germany,soya presence - {allergens}
1,2026.2433,"nuts, nut products and seeds",food,Aflatoxin B1 and totoal on Groundnuts from Arg...,20-03-2026 17:55:22,Netherlands,information notification for attention,serious,Netherlands,INFOSAN,NaN,"Argentina,Netherlands",Argentina,"Aflatoxin B1 - {mycotoxins},aflatoxin total ..."
2,2026.2432,fruits and vegetables,food,Exceeded MRL pesticide Oxamyl in oranges from ...,20-03-2026 17:51:42,Romania,border rejection notification,potentially serious,NaN,INFOSAN,NaN,"Egypt,Romania",Egypt,oxamyl unauthorised substance - {pesticide re...


## Focused EDA: Vietnam, India, Ecuador (SPS/TBT)
This section filters records to the 3 countries and categorizes rejection reasons using keyword rules for SPS and TBT.

In [8]:
import re

TARGET_COUNTRIES = {"vietnam", "india", "ecuador"}
START_DATE = pd.Timestamp("2020-03-30")
END_DATE = pd.Timestamp("2025-12-31")

SPS_KEYWORDS = [
    r"\bsps\b",
    r"sanitary",
    r"phytosanitary",
    r"pesticide",
    r"residue",
    r"mr\s*l",
    r"microbiolog",
    r"salmonella",
    r"listeria",
    r"aflatoxin",
    r"contamin",
    r"pathogen",
    r"veterinary",
    r"animal health",
    r"plant health",
    r"food safety",
    r"mycotoxin",
]

TBT_KEYWORDS = [
    r"\btbt\b",
    r"technical barrier",
    r"technical regulation",
    r"label",
    r"labelling",
    r"packag",
    r"certificate",
    r"documentation",
    r"traceability",
    r"standard",
    r"marking",
    r"conformity",
    r"compliance",
    r"composition",
    r"additive",
    r"quality requirement",
]

def pick_first_column(columns: list[str], include_terms: list[str]) -> str | None:
    for col in columns:
        col_norm = col.lower().strip()
        if any(term in col_norm for term in include_terms):
            return col
    return None

def pick_country_column(columns: list[str]) -> str | None:
    col_map = {c.lower().strip(): c for c in columns}

    # Prefer origin/source country for this analysis.
    for preferred in ["origin", "country of origin", "country_origin", "source country"]:
        if preferred in col_map:
            return col_map[preferred]

    return pick_first_column(columns, ["origin", "country"])

all_columns = df.columns.tolist()
country_col = pick_country_column(all_columns)
date_col = pick_first_column(all_columns, ["date"])

reason_like_cols = [
    c for c in all_columns
    if any(k in c.lower() for k in [
        "reason", "ground", "basis", "subject", "description",
        "hazard", "non-compliance", "non compliance", "notification",
    ])
]

print("Detected country column:", country_col)
print("Detected date column:", date_col)
print("Detected reason-like columns:", reason_like_cols[:8], "..." if len(reason_like_cols) > 8 else "")

if country_col is None:
    raise ValueError("Could not detect a country/origin column. Please set country_col manually.")
if date_col is None:
    raise ValueError("Could not detect a date column. Please set date_col manually.")
if not reason_like_cols:
    raise ValueError("Could not detect reason-related columns. Please set reason_like_cols manually.")

df_tmp = df.copy()
df_tmp["_parsed_date"] = pd.to_datetime(
    df_tmp[date_col],
    errors="coerce",
    dayfirst=True,
    utc=False,
)

country_mask = df_tmp[country_col].astype(str).str.lower().str.strip().isin(TARGET_COUNTRIES)
date_mask = df_tmp["_parsed_date"].between(START_DATE, END_DATE, inclusive="both")
df_focus = df_tmp[country_mask & date_mask].copy()

def combine_reason_text(row: pd.Series, cols: list[str]) -> str:
    parts = [str(row[c]) for c in cols if pd.notna(row[c]) and str(row[c]).strip()]
    return " | ".join(parts)

df_focus["reason_text"] = df_focus.apply(lambda r: combine_reason_text(r, reason_like_cols), axis=1)

sps_pattern = re.compile("|".join(SPS_KEYWORDS), flags=re.IGNORECASE)
tbt_pattern = re.compile("|".join(TBT_KEYWORDS), flags=re.IGNORECASE)

def classify_reason(text: str) -> str:
    txt = str(text)
    has_sps = bool(sps_pattern.search(txt))
    has_tbt = bool(tbt_pattern.search(txt))
    if has_sps and has_tbt:
        return "SPS+TBT"
    if has_sps:
        return "SPS"
    if has_tbt:
        return "TBT"
    return "Other/Unmatched"

df_focus["category"] = df_focus["reason_text"].apply(classify_reason)

print(f"Validation date window: {START_DATE.date()} to {END_DATE.date()}")
print(f"Focused dataset rows: {len(df_focus):,}")
df_focus[[country_col, date_col, "category", "reason_text"]].head(10)

Detected country column: origin
Detected date column: date
Detected reason-like columns: ['subject', 'hazards'] 
Validation date window: 2020-03-30 to 2025-12-31
Focused dataset rows: 2,299


,origin,date,category,reason_text
1034,India,12-12-2025 16:13:55,SPS,Mineral oil residues (MOSH/MOAH) in rice from...
1049,India,12-12-2025 12:03:18,Other/Unmatched,Food supplement with novel food ingredient fro...
1062,India,11-12-2025 16:38:40,SPS,Pesticides residues in food supplement from In...
1067,Vietnam,11-12-2025 15:40:13,SPS,Oxytetracycline in frozen giant shrimp tails (...
1083,India,11-12-2025 11:19:22,Other/Unmatched,Pyrrolizidine alkaloids in black tea from Indi...
1084,India,11-12-2025 10:49:44,SPS,Nitrofurans ((SEM) in shrimps from India | nit...
1085,India,11-12-2025 10:37:10,SPS,chlorpyrifos in basmati rice from India | chlo...
1095,Vietnam,10-12-2025 15:36:07,SPS,Leucomalachite Green in Vannamei shrimps from ...
1096,Vietnam,10-12-2025 14:39:28,Other/Unmatched,Foreign body (metal wire) in ground black pepp...
1102,India,10-12-2025 12:06:08,SPS,Chlorpyrifos in masala spice from India. | chl...


In [9]:
# 1) Overall category distribution for selected countries
overall_summary = (
    df_focus["category"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="count")
)
print("Overall category distribution:")
print(overall_summary.to_string(index=False))

# 2) Country x category pivot
country_category_summary = (
    df_focus.groupby([country_col, "category"])
    .size()
    .reset_index(name="count")
    .sort_values([country_col, "count"], ascending=[True, False])
)
print("\nCountry-category detail:")
print(country_category_summary.to_string(index=False))

country_category_pivot = (
    country_category_summary
    .pivot(index=country_col, columns="category", values="count")
    .fillna(0)
    .astype(int)
)
print("\nCountry-category pivot table:")
print(country_category_pivot)

# 3) Sample unmatched rows for manual keyword refinement
unmatched_samples = df_focus[df_focus["category"] == "Other/Unmatched"][[country_col, "reason_text"]].head(20)
print("\nSample unmatched rows (for keyword tuning):")
print(unmatched_samples.to_string(index=False))

country_category_pivot

Overall category distribution:
       category  count
            SPS   1717
Other/Unmatched    390
            TBT    180
        SPS+TBT     12

Country-category detail:
 origin        category  count
Ecuador             SPS    125
Ecuador Other/Unmatched     39
Ecuador             TBT     27
Ecuador         SPS+TBT      1
  India             SPS   1341
  India Other/Unmatched    252
  India             TBT    119
  India         SPS+TBT      7
Vietnam             SPS    251
Vietnam Other/Unmatched     99
Vietnam             TBT     34
Vietnam         SPS+TBT      4

Country-category pivot table:
category  Other/Unmatched   SPS  SPS+TBT  TBT
origin                                       
Ecuador                39   125        1   27
India                 252  1341        7  119
Vietnam                99   251        4   34

Sample unmatched rows (for keyword tuning):
 origin                                                                                                                

category,Other/Unmatched,SPS,SPS+TBT,TBT
origin,,,,
Ecuador,39,125,1,27
India,252,1341,7,119
Vietnam,99,251,4,34


In [10]:
output_file = Path("EU_Vietnam_India_Ecuador_SPS_TBT.csv")
df_focus.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Saved: {output_file.resolve()}")

Saved: D:\Phuong\Code in Python\KTPT\EU\EU_Vietnam_India_Ecuador_SPS_TBT.csv


## US FDA + EU RASFF Trade Compliance Analysis (Shrimp/Prawns only)
Scope enforced in code:
- Period: 2020-03-30 to 2025-12-31
- Countries: India, Vietnam, Ecuador
- Product scope: only records explicitly mentioning shrimp/prawn(s)
- Classification: SPS, TBT, SPS+TBT, Other (documentation/administrative or unmatched)

In [16]:
from pathlib import Path
import re
import pandas as pd

START = pd.Timestamp("2020-03-30")
END = pd.Timestamp("2025-12-31")
TARGET_COUNTRIES = {"india", "vietnam", "ecuador"}
ISO_TO_COUNTRY = {"IN": "India", "VN": "Vietnam", "EC": "Ecuador"}

SHRIMP_PATTERN = re.compile(r"\b(?:shrimp|prawn)s?\b", flags=re.IGNORECASE)

SPS_PATTERN = re.compile(
    r"salmonella|listeria|vibrio|pathogen|microbiolog|aflatoxin|mycotoxin|"
    r"pesticide|residue|antibiotic|nitrofuran|chloramphenicol|veterinary|"
    r"adulterat|filth|insanitary|contamin|toxin|poison|unsafe|hazard",
    flags=re.IGNORECASE,
 )

TBT_PATTERN = re.compile(
    r"misbrand|label|labelling|net\s*weight|quantity|false|misleading|"
    r"ingredient|composition|standard|packag|marking|conformity|technical",
    flags=re.IGNORECASE,
 )

def load_csv_fallback(path: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(path, encoding="latin1", engine="python", on_bad_lines="skip")

def classify_text(text: str) -> str:
    txt = str(text)
    has_sps = bool(SPS_PATTERN.search(txt))
    has_tbt = bool(TBT_PATTERN.search(txt))
    if has_sps and has_tbt:
        return "SPS+TBT"
    if has_sps:
        return "SPS"
    if has_tbt:
        return "TBT"
    return "Other"

base = Path.cwd()

# -------------------- FDA --------------------
fda_paths = [
    base.parent / "FDA" / "REFUSAL_ENTRY_2019_2023.csv",
    base.parent / "FDA" / "REFUSAL_ENTRY_2024-Feb2026.csv",
]
fda_raw = pd.concat([load_csv_fallback(p) for p in fda_paths], ignore_index=True)

fda = fda_raw.copy()
fda["_date"] = pd.to_datetime(fda["REFUSAL_DATE"], errors="coerce")
fda = fda[fda["_date"].between(START, END, inclusive="both")]
fda = fda[fda["ISO_CNTRY_CODE"].astype(str).str.upper().isin(ISO_TO_COUNTRY.keys())]
fda = fda[fda["PRDCT_CODE_DESC_TEXT"].astype(str).str.contains(SHRIMP_PATTERN, na=False)]

charge_map_paths = [
    base.parent / "FDA" / "ACT_SECTION_CHARGES_1923.csv",
    base.parent / "FDA" / "ACT_SECTION_CHARGES_2426.csv",
]
charge_map = pd.concat([load_csv_fallback(p) for p in charge_map_paths], ignore_index=True)
charge_map["ASC_ID"] = pd.to_numeric(charge_map["ASC_ID"], errors="coerce")
charge_map = charge_map.dropna(subset=["ASC_ID"]).drop_duplicates(subset=["ASC_ID"])
charge_map["ASC_ID"] = charge_map["ASC_ID"].astype(int)
code_to_text = {
    int(r.ASC_ID): f"{r.CHRG_CODE} | {r.CHRG_STMNT_TEXT} | {r.SCTN_NAME}"
    for r in charge_map.itertuples(index=False)
}

def parse_charge_codes(value: str) -> list[int]:
    return [int(x) for x in re.findall(r"\d+", str(value))]

def fda_reason_text(raw_charge: str) -> str:
    codes = parse_charge_codes(raw_charge)
    mapped = [code_to_text[c] for c in codes if c in code_to_text]
    return " || ".join(mapped) if mapped else str(raw_charge)

fda["country"] = fda["ISO_CNTRY_CODE"].astype(str).str.upper().map(ISO_TO_COUNTRY)
fda["reason_text"] = fda["REFUSAL_CHARGES"].apply(fda_reason_text)
fda["category"] = fda["reason_text"].apply(classify_text)
fda_out = fda[["country", "_date", "category", "reason_text"]].copy()
fda_out["source"] = "FDA"

# -------------------- EU --------------------
eu_path = base / "RASFF_raw_data_EU.csv"
eu_raw = load_csv_fallback(eu_path)

eu = eu_raw.copy()
eu["_date"] = pd.to_datetime(eu["date"], errors="coerce", dayfirst=True)
eu = eu[eu["_date"].between(START, END, inclusive="both")]
eu = eu[eu["origin"].astype(str).str.strip().str.lower().isin(TARGET_COUNTRIES)]

eu_product_text = (
    eu["category"].astype(str) + " | " +
    eu["subject"].astype(str) + " | " +
    eu["hazards"].astype(str)
)
eu = eu[eu_product_text.str.contains(SHRIMP_PATTERN, na=False)]
eu["country"] = eu["origin"].astype(str).str.strip().str.title()
eu["reason_text"] = eu[["subject", "hazards"]].fillna("").agg(" | ".join, axis=1)
eu["category"] = eu["reason_text"].apply(classify_text)
eu_out = eu[["country", "_date", "category", "reason_text"]].copy()
eu_out["source"] = "EU"

# -------------------- Consolidated summary --------------------
all_refusals = pd.concat([fda_out, eu_out], ignore_index=True)
all_refusals = all_refusals[all_refusals["country"].isin(["India", "Vietnam", "Ecuador"])]

consolidated = (
    all_refusals
    .groupby(["country", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["SPS", "TBT", "SPS+TBT", "Other"], fill_value=0)
    .reset_index()
 )
consolidated["Total"] = consolidated[["SPS", "TBT", "SPS+TBT", "Other"]].sum(axis=1)
grand_total = consolidated["Total"].sum()
consolidated["Percentage (%)"] = (
    (consolidated["Total"] / grand_total * 100).round(2) if grand_total else 0.0
)
consolidated = consolidated.sort_values("Total", ascending=False).reset_index(drop=True)

source_breakdown = (
    all_refusals.groupby(["source", "country", "category"])
    .size()
    .reset_index(name="count")
    .sort_values(["source", "country", "count"], ascending=[True, True, False])
 )

print(f"Analysis window: {START.date()} to {END.date()}")
print(f"Strict product scope: shrimp/prawns only")
print(f"Total consolidated records: {len(all_refusals):,}")
print("\nConsolidated table (FDA + EU):")
display(consolidated)

print("\nSource-level diagnostic breakdown:")
display(source_breakdown.head(30))

all_refusals.to_csv(base / "EU_FDA_ShrimpPrawns_Refusals_2020_2025_detailed.csv", index=False, encoding="utf-8-sig")
consolidated.to_csv(base / "EU_FDA_ShrimpPrawns_Refusals_2020_2025_summary.csv", index=False, encoding="utf-8-sig")
print("Saved detailed and summary CSV files in current folder.")

C:\Users\voila\AppData\Local\Temp\ipykernel_716\4013543371.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fda["_date"] = pd.to_datetime(fda["REFUSAL_DATE"], errors="coerce")


Analysis window: 2020-03-30 to 2025-12-31
Strict product scope: shrimp/prawns only
Total consolidated records: 737

Consolidated table (FDA + EU):


category,country,SPS,TBT,SPS+TBT,Other,Total,Percentage (%)
0,India,454,13,5,2,474,64.31
1,Ecuador,120,3,0,15,138,18.72
2,Vietnam,113,2,2,8,125,16.96



Source-level diagnostic breakdown:


,source,country,category,count
1,EU,Ecuador,SPS,76
0,EU,Ecuador,Other,15
2,EU,Ecuador,TBT,3
4,EU,India,SPS,39
5,EU,India,TBT,4
3,EU,India,Other,2
7,EU,Vietnam,SPS,31
6,EU,Vietnam,Other,8
8,FDA,Ecuador,SPS,44
9,FDA,India,SPS,415


Saved detailed and summary CSV files in current folder.


## EU RASFF Strict Analysis: Shrimp/Prawns (2020-03-30 to 2025-12-31)
This section enforces strict constraints:
- Product scope: only shipments explicitly mentioning shrimp/prawn(s)
- Countries: India, Vietnam, Ecuador
- Date range: 2020-03-30 to 2025-12-31
- Categories: SPS, TBT, SPS+TBT, Other

In [17]:
from pathlib import Path
import re
import pandas as pd

# -------------------- Strict analysis parameters --------------------
EU_START = pd.Timestamp("2020-03-30")
EU_END = pd.Timestamp("2025-12-31")
EU_TARGET_COUNTRIES = {"india", "vietnam", "ecuador"}

SHRIMP_PRAWN_PATTERN = re.compile(r"\b(?:shrimp|prawn)s?\b", flags=re.IGNORECASE)

# SPS: biological/chemical hazard style terms
EU_SPS_PATTERN = re.compile(
    r"salmonella|listeria|vibrio|pathogen|microbiolog|aflatoxin|mycotoxin|"
    r"pesticide|residue|antibiotic|nitrofuran|chloramphenicol|veterinary|"
    r"filth|insanitary|contamin|toxin|poison|unsafe|hazard|parasite|heavy\s*metal",
    flags=re.IGNORECASE,
 )

# TBT: technical regulation and labeling/standard style terms
EU_TBT_PATTERN = re.compile(
    r"misbrand|label|labelling|packag|net\s*weight|quantity|false|misleading|"
    r"ingredient|composition|marking|conformity|technical|standard|non\s*compliance|"
    r"certificat|traceability|declaration",
    flags=re.IGNORECASE,
 )

def _load_csv_with_fallback(path: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(path, encoding="latin1", engine="python", on_bad_lines="skip")

def _classify_refusal_reason(text: str) -> str:
    txt = str(text)
    has_sps = bool(EU_SPS_PATTERN.search(txt))
    has_tbt = bool(EU_TBT_PATTERN.search(txt))
    if has_sps and has_tbt:
        return "SPS+TBT"
    if has_sps:
        return "SPS"
    if has_tbt:
        return "TBT"
    return "Other"

eu_file = Path("RASFF_raw_data_EU.csv")
eu_raw_strict = _load_csv_with_fallback(eu_file)

# Harmonize required columns
eu_raw_strict["_date"] = pd.to_datetime(eu_raw_strict["date"], errors="coerce", dayfirst=True)
eu_raw_strict["_country_norm"] = eu_raw_strict["origin"].astype(str).str.strip().str.lower()

# Build product scope text and keep only shrimp/prawn rows
_product_scope_text = (
    eu_raw_strict["category"].astype(str) + " | " +
    eu_raw_strict["subject"].astype(str) + " | " +
    eu_raw_strict["hazards"].astype(str)
 )

eu_filtered = eu_raw_strict[
    eu_raw_strict["_date"].between(EU_START, EU_END, inclusive="both")
    & eu_raw_strict["_country_norm"].isin(EU_TARGET_COUNTRIES)
    & _product_scope_text.str.contains(SHRIMP_PRAWN_PATTERN, na=False)
].copy()

# Build reason text for classification
eu_filtered["reason_text"] = eu_filtered[["subject", "hazards"]].fillna("").agg(" | ".join, axis=1)
eu_filtered["category_class"] = eu_filtered["reason_text"].apply(_classify_refusal_reason)
eu_filtered["country"] = eu_filtered["_country_norm"].str.title()

# Consolidated table for the 3 countries
eu_consolidated = (
    eu_filtered.groupby(["country", "category_class"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["SPS", "TBT", "SPS+TBT", "Other"], fill_value=0)
    .reindex(index=["India", "Vietnam", "Ecuador"], fill_value=0)
    .reset_index()
 )
eu_consolidated["Total"] = eu_consolidated[["SPS", "TBT", "SPS+TBT", "Other"]].sum(axis=1)

_total_rows = eu_consolidated["Total"].sum()
eu_consolidated["Percentage (%)"] = (
    (eu_consolidated["Total"] / _total_rows * 100).round(2) if _total_rows else 0.0
 )

# Save filtered datasets
strict_detailed_out = Path("EU_RASFF_ShrimpPrawns_2020_2025_filtered.csv")
strict_summary_out = Path("EU_RASFF_ShrimpPrawns_2020_2025_summary.csv")

selected_cols = [
    "country",
    "_date",
    "category_class",
    "subject",
    "hazards",
    "reason_text",
    "reference",
    "notification type",
    "distribution status",
 ]
existing_selected_cols = [c for c in selected_cols if c in eu_filtered.columns]

eu_filtered[existing_selected_cols].to_csv(strict_detailed_out, index=False, encoding="utf-8-sig")
eu_consolidated.to_csv(strict_summary_out, index=False, encoding="utf-8-sig")

print(f"Analysis window: {EU_START.date()} to {EU_END.date()}")
print("Product scope: shrimp/prawns only")
print(f"Filtered EU records: {len(eu_filtered):,}")
print("\nConsolidated table (India, Vietnam, Ecuador):")
display(eu_consolidated)

print("\nSaved files:")
print(strict_detailed_out.resolve())
print(strict_summary_out.resolve())

Analysis window: 2020-03-30 to 2025-12-31
Product scope: shrimp/prawns only
Filtered EU records: 178

Consolidated table (India, Vietnam, Ecuador):


category_class,country,SPS,TBT,SPS+TBT,Other,Total,Percentage (%)
0,India,39,4,0,2,45,25.28
1,Vietnam,31,0,0,8,39,21.91
2,Ecuador,76,3,0,15,94,52.81



Saved files:
D:\Phuong\Code in Python\KTPT\EU\EU_RASFF_ShrimpPrawns_2020_2025_filtered.csv
D:\Phuong\Code in Python\KTPT\EU\EU_RASFF_ShrimpPrawns_2020_2025_summary.csv
